# Probability reliability and edge cases after 2.5× oversampling

A classifier can produce the right **number** of each label while attaching poor probabilities to individual rows. This notebook therefore freezes the 2.5× multiplier and asks four different questions using the saved local-test probabilities—no new models are fitted.

1. **Reliability:** when the model says 80%, is that event observed about 80% of the time?
2. **Winning margin:** how far ahead is the selected class from the runner-up?
3. **Component disagreement:** do the models inside a blend choose the same class?
4. **Edge groups:** which confident errors, close calls, repair false alarms and repair misses changed?

## Course-aligned ten-step lifecycle

| Step | Treatment |
|---:|---|
| 1. Define the goal and scope | Diagnose probability quality and consequential error groups after freezing 2.5× oversampling. |
| 2. Gather the data | Reuse saved original and 2.5× local-test probabilities plus the frozen labelled local test. |
| 3. Explore the data | Compare predicted probabilities, confidence, margins, disagreement and error groups. |
| 4. Clean and preprocess the data | Not applicable: no predictor data is refitted here. |
| 5. Select and engineer features | Not applicable: feature policy remains frozen. |
| 6. Define the machine-learning task | Diagnose three-class probability estimates and nominal decisions. |
| 7. Partition the data | Use only the already-opened, fingerprinted local test for retrospective diagnosis. |
| 8. Select and train candidate methods | No new fitting; multiplier and recipes are frozen. |
| 9. Evaluate and interpret the results | Report calibration error, log loss, Brier score, margins, disagreement and edge groups. |
| 10. Deploy and iterate | Record implications; prohibit further multiplier tuning in this session. |

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

stage_directory = next(
    candidate for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / 'data' / 'TrainingSetValues.csv').is_file()
)
result_path = (
    stage_directory.parent
    / '.runtime'
    / 'oversampled-reliability'
    / 'results.json'
)
results = json.loads(result_path.read_text(encoding='utf-8'))
recipe_labels = {
    'random-forest-histogram-boosting': '50% RF + 50% histogram',
    '55-xgboost-depth-8-child-1-current-one-hot-45-random-forest': (
        '55% XGB depth 8 child 1 + 45% RF'
    ),
    '60-xgboost-depth-6-7-8-local-bag-40-random-forest': (
        '60% XGB depth bag + 40% RF'
    ),
}
selected_recipe = (
    '55-xgboost-depth-8-child-1-current-one-hot-45-random-forest'
)
print(
    f"Loaded saved probabilities for {results['local_test_rows']:,} local-test rows; "
    f"new model fits: {results['new_model_fits']}."
)

Loaded saved probabilities for 11,880 local-test rows; new model fits: 0.


## Overall probability quality

**Log loss** heavily penalises confident probability assigned to the wrong class. **Brier score** is the mean squared error of all three class probabilities. Lower is better for both. **Expected calibration error (ECE)** groups winning confidences into ten bands and measures the weighted gap between confidence and accuracy; it is an intuitive summary, not a complete proof of calibration.

In [2]:
quality_rows = []
for regime, recipes in results['regimes'].items():
    for recipe, record in recipes.items():
        quality = record['quality']
        disagreement = record['component_disagreement']
        quality_rows.append(
            {
                'recipe': recipe_labels[recipe],
                'training': regime,
                'accuracy': quality['accuracy'],
                'log loss': quality['log_loss'],
                'Brier': quality['multiclass_brier'],
                'top-label ECE': quality['top_label_ece'],
                'mean confidence': quality['mean_confidence'],
                'mean winning margin': quality['mean_winning_margin'],
                'close-decision share': quality['low_margin_share'],
                'component disagreement': (
                    disagreement['non_unanimous_share']
                ),
            }
        )
quality_comparison = pd.DataFrame(quality_rows).set_index(
    ['recipe', 'training']
)
display(quality_comparison.style.format('{:.3f}'))

,,accuracy,log loss,Brier,top-label ECE,mean confidence,mean winning margin,close-decision share,component disagreement
recipe,training,,,,,,,,
50% RF + 50% histogram,original training,0.809,0.471,0.271,0.012,0.800,0.635,0.065,0.122
55% XGB depth 8 child 1 + 45% RF,original training,0.807,0.468,0.269,0.006,0.810,0.652,0.059,0.098
60% XGB depth bag + 40% RF,original training,0.808,0.468,0.269,0.004,0.807,0.648,0.060,0.118
50% RF + 50% histogram,2.5x repair training,0.801,0.482,0.278,0.015,0.788,0.618,0.069,0.139
55% XGB depth 8 child 1 + 45% RF,2.5x repair training,0.803,0.477,0.276,0.013,0.798,0.635,0.065,0.116
60% XGB depth bag + 40% RF,2.5x repair training,0.803,0.478,0.276,0.009,0.795,0.630,0.069,0.134


## Hard-label balance versus probability balance

The 2.5× recipes produce roughly the right final **hard-label share** for repair, but the following classwise table asks a different question: what is the average probability mass assigned to each class before selecting the largest value? For a representative blend, oversampling moves average repair probability above its observed frequency.

In [3]:
classwise_rows = []
for regime in results['regimes']:
    records = results['regimes'][regime][selected_recipe]['classwise']
    for record in records:
        classwise_rows.append({'training': regime, **record})
classwise_comparison = pd.DataFrame(classwise_rows).set_index(
    ['class', 'training']
)
display(classwise_comparison.style.format('{:.3f}'))

,,natural_share,mean_probability,probability_minus_share,one_vs_rest_brier,one_vs_rest_ece
class,training,,,,,
functional,original training,0.543,0.543,-0.000,0.120,0.011
functional needs repair,original training,0.073,0.073,0.000,0.050,0.009
non functional,original training,0.384,0.385,0.000,0.099,0.014
functional,2.5x repair training,0.543,0.523,-0.020,0.123,0.020
functional needs repair,2.5x repair training,0.073,0.101,0.029,0.054,0.029
non functional,2.5x repair training,0.384,0.376,-0.009,0.099,0.013


## Reading a reliability table

For each repair-probability band, compare `mean_probability` with `observed_frequency`. A reliable 0.6–0.7 band should contain repair labels about 60–70% of the time. Large positive `gap` means the event happens more often than predicted; a large negative gap means the probability is overconfident. Empty or tiny bands should not be over-interpreted.

In [4]:
repair_reliability = {}
for regime in results['regimes']:
    frame = pd.DataFrame(
        results['regimes'][regime][selected_recipe]['repair_reliability']
    ).set_index('bin')
    repair_reliability[regime] = frame
    print(regime)
    display(
        frame.loc[frame['rows'].gt(0)].style.format(
            {
                'mean_probability': '{:.1%}',
                'observed_frequency': '{:.1%}',
                'gap': '{:+.1%}',
                'absolute_gap': '{:.1%}',
            }
        )
    )

original training


,rows,mean_probability,observed_frequency,gap,absolute_gap
bin,,,,,
0.0–0.1,9688,1.9%,2.1%,+0.3%,0.3%
0.1–0.2,901,14.3%,16.6%,+2.4%,2.4%
0.2–0.3,444,24.5%,25.2%,+0.7%,0.7%
0.3–0.4,285,34.7%,30.5%,-4.1%,4.1%
0.4–0.5,161,45.1%,34.2%,-10.9%,10.9%
0.5–0.6,132,54.7%,45.5%,-9.2%,9.2%
0.6–0.7,93,64.6%,66.7%,+2.1%,2.1%
0.7–0.8,69,74.4%,69.6%,-4.8%,4.8%
0.8–0.9,80,84.7%,71.2%,-13.4%,13.4%


2.5x repair training


,rows,mean_probability,observed_frequency,gap,absolute_gap
bin,,,,,
0.0–0.1,8890,2.2%,1.6%,-0.6%,0.6%
0.1–0.2,1135,14.3%,10.1%,-4.2%,4.2%
0.2–0.3,560,24.4%,17.3%,-7.1%,7.1%
0.3–0.4,375,34.7%,25.9%,-8.8%,8.8%
0.4–0.5,268,44.9%,32.1%,-12.9%,12.9%
0.5–0.6,194,54.7%,30.9%,-23.8%,23.8%
0.6–0.7,142,64.8%,38.7%,-26.1%,26.1%
0.7–0.8,121,75.0%,61.2%,-13.9%,13.9%
0.8–0.9,118,85.0%,66.1%,-18.9%,18.9%


## Edge groups and the price of better repair recall

A **winning margin** below ten percentage points means the first and second choices are close; those rows behave almost like coin tosses and are natural inspection candidates. **Component disagreement** is a related signal: one member of the blend chooses another class. Neither proves that a row is intrinsically ambiguous, but both locate cases where the model's decision is fragile.

In [5]:
edge_frames = []
for regime in results['regimes']:
    frame = pd.DataFrame(
        results['regimes'][regime][selected_recipe]['edge_cases']
    )
    frame.insert(0, 'training', regime)
    edge_frames.append(frame)
edge_comparison = pd.concat(edge_frames).set_index(['group', 'training'])
display(
    edge_comparison.style.format(
        {'share': '{:.2%}', 'mean_confidence': '{:.1%}', 'accuracy': '{:.1%}'}
    )
)

,,rows,share,mean_confidence,accuracy
group,training,,,,
high-confidence correct (>=80%),original training,6545,55.09%,93.4%,100.0%
high-confidence error (>=80%),original training,484,4.07%,88.1%,0.0%
close decision (margin <10 points),original training,699,5.88%,48.2%,44.2%
repair predicted correctly,original training,283,2.38%,68.6%,100.0%
repair false alarm,original training,224,1.89%,58.0%,0.0%
repair missed as functional,original training,441,3.71%,68.7%,0.0%
repair missed as non-functional,original training,139,1.17%,63.0%,0.0%
high-confidence correct (>=80%),2.5x repair training,6251,52.62%,93.2%,100.0%
high-confidence error (>=80%),2.5x repair training,444,3.74%,88.0%,0.0%


## Interpretation and stop decision

The 2.5× XGBoost/Random Forest blend converts 94 additional repair rows from misses to correct repair predictions, but creates 207 additional repair false alarms. Its overall accuracy falls by only 0.40 points because other class movements partly compensate. Winning-confidence calibration remains reasonably close overall—ECE rises from 0.56% to 1.34%—but repair probability is systematically inflated: mean repair probability rises from 7.28% to 10.12% against a 7.26% observed share. Repair ECE rises from 0.94% to 2.86%, and both log loss and Brier score worsen.

The result is therefore attractive for **hard classifications under a stronger repair-recall preference**, not for interpreting the raw probabilities as repair risk. Probability calibration would have to be selected and tested through fresh nested development folds; calibrating against this local test would leak. Component disagreement rises from 9.76% to 11.62%, while close decisions rise from 5.88% to 6.54%, giving useful error-analysis cohorts.

**Stop decision:** 2.5× is frozen as the final multiplier examined in this session. No 2.4×, 2.6× or calibration adjustment will be tried against the observed local test. A later iteration may compare pre-declared multiplier candidates and calibration methods solely inside development cross-validation.

In [6]:
assert results['new_model_fits'] == 0
assert results['multiplier_frozen'] == 2.5
assert len(quality_comparison) == 6
assert len(classwise_comparison) == 6
assert sum(frame['rows'].sum() for frame in repair_reliability.values()) == 2 * results['local_test_rows']
assert edge_comparison.index.is_unique
print('Reliability, disagreement and edge-case records verified.')

Reliability, disagreement and edge-case records verified.
